## 5. Висновки

### Dataset VisDrone чому

Почитавши про Visicom, яке працює з ГІС системами і міськими моделями, було прийнято рішення шукати датасет в напрямку знімків міста. **VisDrone2019-DET** зйомка з дрона, 14 міст, реальний міський трафік.

Датасет уже в YOLO-форматі через Ultralytics, де має 10 класів, присутні train/val/test
Під вимоги тестового завдання підходить ідеально.

### Що вийшло добре

На значенні **val** після 80 епох показано: mAP50 близько *0.30*, precision *0.43*, recall *0.33*. На **test** нижче (mAP50 *0.26*).

Класи **car** і **bus** тягнуть на себе результат, де на val у car mAP50 близько *0.73*, на test близько *0.66*. На нещільних сценах з великими авто bbox виглядають стабільно, це видно і на прикладах *good*.

Loss на train і val падав без різкого розходження, ранній стоп не спрацював, тож модель ще підростала до кінця. Тобто базовий пайплайн такий EDA -> train -> val/test -> predict відпрацював правильно.

### Де модель помиляється

Найгірші класи на test: **bicycle** (0.05), **people** (0.10), **awning-tricycle** (0.13). Натовпи дрібних пішоходів, далекі об'єкти, велосипеди на яких модель або пропускає, або малює зайве.

`people` і `pedestrian` плутаються між собою, це видно і в метриках, і на confusion matrix. На bad прикладах типова картина: щільна сцена та маленькі bbox = низький recall.

Це не "непередбачувана ситуація", адже на EDA вже було близько **97.5%** дрібних об'єктів і сильний дисбаланс класів. Гіпотеза з аналізу датасету підтвердилась на практиці.

### Тож

1. **Датасет** - VisDrone важкий саме через small objects і рідкісні класи;
2. **Модель** - YOLOv8n найлегша; ємності мережі мало для дрібних деталей;
3. **imgsz=640** - багато об'єктів на кадрі займають кілька пікселів після ресайзу;
4. **Дисбаланс** - car домінує, bicycle / awning-tricycle майже не бачать градієнт.

### Що б змінив / додав з більшим часом

Натренувати модельку довше можна, однак є кілька напрямів, які логічно випливають з того, що вже побачили:

**Роздільна здатність**
- підняти *imgsz до 800*
- увімкнути *multi-scale* training, щоб модель могла бачити об'єкти різного масштабу

**data/classes**
- об'єднати `people` + `pedestrian` в один клас, це додає менше плутанини
- відфільтрувати / переглянути розмітку на порожніх або сумнівних кадрах (як то було з краном у `others` на етапі EDA)

**Тренування**
- довше тренування (120 епох) або окремий run з іншим lr
- сильніше штрафувати помилки на рідкісних класах

**Іншу ідеї**
- різати велике аерофото на тайли, потім зливати bbox назад
- порівняти день / ніч. На датасеті VisDrone є різні умови зйомки, можна зробити окремий error analysis по цих сценах
- додати *heatmaps помилок* по кадру де візуально видно, що модель буквально сліпне на далекій перспективі

### Підсумок

VisDrone як релевантний до GIS/дронів датасет, є повний цикл від EDA до test і error analysis. YOLOv8n на 80 епохах дає mAP50 близько 0.30 на val і 0.26 на test: добре тримає великі авто, слабко — дрібні й рідкісні класи. Узгоджується з EDA. Далі логічно йти в більшу модель або вищу роздільність, чистку класів.